In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

In [ ]:
REGION_TO_ELECTRODES = {
    'Middle_Finger': ['E1', 'E2', 'E3'],
    'Hand': ['E4', 'E5', 'E6', 'E7'],
    'Forearm': ['E8', 'E9', 'E10', 'E11', 'E12', 'E13'],
    'Arm': ['E14', 'E15', 'E16', 'E17', 'E18', 'E19', 'E20']
}

ELECTRODE_TO_REGION = {
    electrode: region
    for region, electrodes in REGION_TO_ELECTRODES.items()
    for electrode in electrodes
}

REGION_TO_LABEL = {
    'Middle_Finger': 0,
    'Hand': 1,
    'Forearm': 2,
    'Arm': 3
}

LABEL_TO_REGION = {v: k for k, v in REGION_TO_LABEL.items()}

print("Region-to-Electrode Mapping:")
for region, electrodes in REGION_TO_ELECTRODES.items():
    print(f"  {region} (Class {REGION_TO_LABEL[region]}): {electrodes} ({len(electrodes)} electrodes)")

In [ ]:
import re
from pathlib import Path

BIDS_ROOT = Path(r"../../data/BIDS-somatosensory/BIDS-somatosensory")
DERIVATIVES = BIDS_ROOT / "derivatives" / "fmriprep"
EVENTS_DIR = BIDS_ROOT / "events"

session = "ses-01"
task = "task-S1Map"
space = "MNI152NLin2009cAsym"
n_runs_per_subject = 4

HRF_DELAY = 6.0
WINDOW = 1

# Keep only subjects present in both derivatives and events
derivative_subjects = sorted(
    d.name for d in DERIVATIVES.iterdir() if d.is_dir() and d.name.startswith("sub-")
)

event_subjects = set()
for ev_path in EVENTS_DIR.glob(f"sub-*_{session}_{task}_run-*_events.tsv"):
    m = re.match(r"^(sub-[^_]+)_", ev_path.name)
    if m:
        event_subjects.add(m.group(1))
event_subjects = sorted(event_subjects)

subjects = sorted(set(derivative_subjects) & set(event_subjects))

print(f"Subjects in derivatives: {len(derivative_subjects)}")
print(f"Subjects in events:      {len(event_subjects)}")
print(f"Subjects included:       {len(subjects)}")
print(subjects)

In [ ]:
def resolve_bold_path(subject, run, space):
    func_dir = DERIVATIVES / subject / session / "func"
    candidates = [
        func_dir / f"{subject}_{session}_{task}_run-{run}_space-{space}_desc-preproc_bold.nii",
        func_dir / f"{subject}_{session}_{task}_run-{run}_space-{space}_desc-preproc_bold.nii.gz",
    ]
    for p in candidates:
        if p.exists():
            return p
    return None

In [ ]:
def get_tr(subject, run=1, default_tr=2.0):
    func_dir = DERIVATIVES / subject / session / "func"

    json_candidates = [
        func_dir / f"{subject}_{session}_{task}_run-{run}_space-{space}_desc-preproc_bold.json",
        func_dir / f"{subject}_{session}_{task}_run-{run}_desc-preproc_bold.json",
    ]

    for bold_json in json_candidates:
        if bold_json.exists():
            with open(bold_json, "r", encoding="utf-8") as f:
                meta = json.load(f)

            if "RepetitionTime" in meta and meta["RepetitionTime"] is not None:
                return float(meta["RepetitionTime"])

            print(
                f"Warning: RepetitionTime missing in {bold_json.name} for {subject}. "
                f"Using default TR={default_tr} s."
            )
            return float(default_tr)

    print(
        f"Warning: No TR JSON found for {subject} (run {run}). "
        f"Using default TR={default_tr} s."
    )
    return float(default_tr)

In [ ]:
tr_by_subject = {sub: get_tr(sub, run=1, default_tr=2.0) for sub in subjects}
unique_trs = sorted(set(tr_by_subject.values()))

print(f"TR values across subjects: {unique_trs}")
if len(unique_trs) > 1:
    print("WARNING: TR is not identical across subjects; per-subject TR will be used.")
else:
    print(f"TR is consistent: {unique_trs[0]} s")

In [ ]:
import pandas as pd

def load_subject_events(subject):
    all_events = []
    required_cols = {"onset", "duration", "trial_type"}

    for run in range(1, n_runs_per_subject + 1):
        ev_path = EVENTS_DIR / f"{subject}_{session}_{task}_run-{run}_events.tsv"
        if not ev_path.exists():
            raise FileNotFoundError(f"Missing events file: {ev_path}")

        df = pd.read_csv(ev_path, sep="\t")
        df.columns = [str(c).strip().lower() for c in df.columns]

        missing = required_cols - set(df.columns)
        if missing:
            raise ValueError(
                f"{ev_path.name} missing columns {sorted(missing)}. "
                f"Found {list(df.columns)}"
            )

        df["run"] = run
        all_events.append(df)

    events_df = pd.concat(all_events, ignore_index=True)
    stim_events = events_df[~events_df["trial_type"].isin(["Baseline", "Jitter"])].copy()
    stim_events["region"] = stim_events["trial_type"].map(ELECTRODE_TO_REGION)

    unmapped = int(stim_events["region"].isna().sum())
    if unmapped > 0:
        bad_labels = sorted(stim_events.loc[stim_events["region"].isna(), "trial_type"].unique())
        raise ValueError(f"Unmapped trial_type values for {subject}: {bad_labels}")

    return events_df, stim_events

In [ ]:
RESULTS_BASE_DIR = Path("results/4_Classes_ANN_MultiSubject")
RESULTS_BASE_DIR.mkdir(parents=True, exist_ok=True)

print(f"Base results folder: {RESULTS_BASE_DIR.resolve()}")
print(f"Subjects to process ({len(subjects)}): {subjects}")

In [ ]:
from nilearn.datasets import fetch_atlas_destrieux_2009
from nilearn.image import load_img, index_img, resample_to_img, new_img_like
from nilearn.maskers import NiftiMasker

atlas = fetch_atlas_destrieux_2009()
atlas_img = load_img(atlas.maps)
atlas_data = atlas_img.get_fdata()

s1_indices = [
    i for i, lab in enumerate(atlas.labels)
    if "L G_postcentral" in str(lab) and i != 0
]

mask_data = np.isin(atlas_data, s1_indices).astype("uint8")
s1_mask = new_img_like(atlas_img, mask_data)

print("Selected atlas regions:")
for i in s1_indices:
    print(f"  - {atlas.labels[i]}")

In [ ]:
import torch
import torch.nn as nn

class SomatotopicANN(nn.Module):
    def __init__(self, input_size: int, dropout: float):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_size, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 16),
            nn.BatchNorm1d(16),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(16, 4),
        )

    def forward(self, x):
        return self.net(x)

In [ ]:
LEARNING_RATE  = 0.001
WEIGHT_DECAY   = 0.001
DROPOUT        = 0.3
BATCH_SIZE     = 32
MAX_EPOCHS     = 500
PATIENCE       = 100  

print(
    f"ANN [subject-specific input]→64→16→4 | "
    f"lr={LEARNING_RATE} | wd={WEIGHT_DECAY} | "
    f"dropout={DROPOUT} | batch={BATCH_SIZE} | max_epochs={MAX_EPOCHS}"
)

In [ ]:
import copy
import time
import pickle
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    precision_recall_fscore_support,
)
import matplotlib.pyplot as plt
import seaborn as sns
from nilearn.image import load_img, index_img, mean_img, resample_to_img
from nilearn.maskers import NiftiMasker

all_subjects_results = []

for subject in subjects:
    print(f"\n{'='*60}\n{subject}")
    t0 = time.time()

    subj_dir = RESULTS_BASE_DIR / subject
    figures_dir = subj_dir / "figures"
    models_dir = subj_dir / "models"
    figures_dir.mkdir(parents=True, exist_ok=True)
    models_dir.mkdir(parents=True, exist_ok=True)

    tr_subject = tr_by_subject[subject]

    # Subject-specific first run + masker
    first_run_path = resolve_bold_path(subject, run=1, space=space)
    if first_run_path is None:
        print(f"  Skipping {subject}: missing run-1 preproc BOLD in space {space}")
        continue

    first_run_img = load_img(str(first_run_path))
    ref_img = index_img(first_run_img, 0)

    s1_mask_resampled = resample_to_img(s1_mask, ref_img, interpolation="nearest")
    masker = NiftiMasker(mask_img=s1_mask_resampled, standardize=False)
    masker.fit(first_run_img)

    # 1) Load events
    events_df, stim_events = load_subject_events(subject)
    print(f"  Stimulation events: {len(stim_events)}")

    # 2) Extract subject feature vectors
    X_list, y_list, run_list = [], [], []

    for run in range(1, n_runs_per_subject + 1):
        bold_path = resolve_bold_path(subject, run=run, space=space)
        if bold_path is None:
            print(f"  Missing BOLD run {run} for {subject} in space {space}, skipping run.")
            continue

        bold_img = load_img(str(bold_path))
        run_evs = stim_events[stim_events["run"] == run]

        for _, event in run_evs.iterrows():
            vol_center = int(np.round((event["onset"] + HRF_DELAY) / tr_subject))
            vol_indices = [vol_center - WINDOW, vol_center, vol_center + WINDOW]

            if all(0 <= v < bold_img.shape[3] for v in vol_indices):
                averaged_img = mean_img(index_img(bold_img, vol_indices), copy_header=True)
                feat = masker.transform(averaged_img).ravel()
                X_list.append(feat)
                y_list.append(event["region"])
                run_list.append(run)

    if len(X_list) == 0:
        print(f"  Skipping {subject}: no usable samples after extraction.")
        continue

    X = np.vstack(X_list).astype(np.float32)
    y = np.array([REGION_TO_LABEL[r] for r in y_list])
    run_labels = np.array(run_list)

    print(f"  Feature matrix: {X.shape}")
    if X.ndim != 2:
        raise ValueError(f"Expected 2D X, got shape {X.shape}")

    input_size_subject = X.shape[1]
    if input_size_subject <= 1:
        raise ValueError(
            f"Unexpected feature width={input_size_subject} for {subject}. "
            "Check masker/ROI extraction."
        )

    cw = compute_class_weight("balanced", classes=np.unique(y), y=y)
    cw_tensor = torch.FloatTensor(cw)

    # 3) Leave-One-Run-Out CV
    fold_results = []
    best_bal_acc = 0.0
    best_state = None
    best_scaler = None

    for test_run in range(1, n_runs_per_subject + 1):
        train_mask = run_labels != test_run
        test_mask = run_labels == test_run

        X_tr, X_te = X[train_mask], X[test_mask]
        y_tr, y_te = y[train_mask], y[test_mask]

        if len(X_te) == 0:
            print(f"  Fold {test_run}: no test samples, skipping fold.")
            continue
        if len(X_tr) == 0:
            print(f"  Fold {test_run}: no train samples, skipping fold.")
            continue

        scaler = StandardScaler()
        X_tr_s = scaler.fit_transform(X_tr)
        X_te_s = scaler.transform(X_te)

        X_te_t = torch.FloatTensor(X_te_s)

        loader = DataLoader(
            TensorDataset(torch.FloatTensor(X_tr_s), torch.LongTensor(y_tr)),
            batch_size=BATCH_SIZE,
            shuffle=True,
        )

        torch.manual_seed(RANDOM_SEED)
        model = SomatotopicANN(input_size=input_size_subject, dropout=DROPOUT)
        optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
        criterion = nn.CrossEntropyLoss(weight=cw_tensor)
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode="max",
            factor=0.5,
            patience=PATIENCE,
            min_lr=LEARNING_RATE,
        )

        best_fold_bal = 0.0
        best_fold_state = None
        no_improve = 0

        for epoch in range(MAX_EPOCHS):
            model.train()
            for bX, by in loader:
                optimizer.zero_grad()
                criterion(model(bX), by).backward()
                optimizer.step()

            model.eval()
            with torch.no_grad():
                preds = model(X_te_t).argmax(dim=1).numpy()

            bal = balanced_accuracy_score(y_te, preds) * 100
            scheduler.step(bal)

            if bal > best_fold_bal:
                best_fold_bal = bal
                best_fold_state = copy.deepcopy(model.state_dict())
                no_improve = 0
            else:
                no_improve += 1

            if no_improve >= PATIENCE:
                break

        if best_fold_state is None:
            print(f"  Fold {test_run}: no valid best state, skipping fold.")
            continue

        model.load_state_dict(best_fold_state)
        model.eval()
        with torch.no_grad():
            y_pred = model(X_te_t).argmax(dim=1).numpy()

        acc = accuracy_score(y_te, y_pred)
        bal = balanced_accuracy_score(y_te, y_pred)
        cm = confusion_matrix(y_te, y_pred, labels=list(range(4)))
        prec, rec, f1, _ = precision_recall_fscore_support(
            y_te, y_pred, labels=list(range(4)), average=None, zero_division=0
        )

        fold_results.append(
            {"fold": test_run, "acc": acc, "bal": bal, "cm": cm, "prec": prec, "rec": rec, "f1": f1}
        )
        print(f"  Fold {test_run}: acc={acc*100:.1f}%  bal={bal*100:.1f}%  ({epoch+1} ep)")

        if bal > best_bal_acc:
            best_bal_acc = bal
            best_state = copy.deepcopy(best_fold_state)
            best_scaler = copy.deepcopy(scaler)

    if len(fold_results) == 0:
        print(f"  Skipping {subject}: all folds were empty/invalid.")
        continue

    if best_state is None or best_scaler is None:
        print(f"  Skipping {subject}: no best model/scaler available.")
        continue

    # 4) Aggregate folds
    cm_total = sum(r["cm"] for r in fold_results)
    mean_acc = np.mean([r["acc"] for r in fold_results])
    std_acc = np.std([r["acc"] for r in fold_results])
    mean_bal = np.mean([r["bal"] for r in fold_results])
    std_bal = np.std([r["bal"] for r in fold_results])
    mean_prec = np.mean([r["prec"] for r in fold_results], axis=0)
    mean_rec = np.mean([r["rec"] for r in fold_results], axis=0)
    mean_f1 = np.mean([r["f1"] for r in fold_results], axis=0)

    print(
        f"  -> acc={mean_acc*100:.1f}+-{std_acc*100:.1f}%  "
        f"bal={mean_bal*100:.1f}+-{std_bal*100:.1f}%  "
        f"time={time.time()-t0:.0f}s"
    )

    # 5) Save model/scaler/cm
    torch.save(best_state, models_dir / "best_model.pt")
    np.save(models_dir / "confusion_matrix.npy", cm_total)
    with open(models_dir / "scaler.pkl", "wb") as f:
        pickle.dump(best_scaler, f)

    # 6) Subject confusion matrix figure
    class_names = [LABEL_TO_REGION[i] for i in range(4)]
    fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(
        cm_total,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=class_names,
        yticklabels=class_names,
        ax=ax,
    )
    ax.set_title(f"{subject}\nbal.acc = {mean_bal*100:.1f}%")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    plt.tight_layout()
    plt.savefig(figures_dir / "confusion_matrix.png", dpi=150)
    plt.close()

    all_subjects_results.append(
        {
            "subject": subject,
            "mean_accuracy": mean_acc,
            "std_accuracy": std_acc,
            "mean_balanced_accuracy": mean_bal,
            "std_balanced_accuracy": std_bal,
            "macro_precision": float(np.mean(mean_prec)),
            "macro_recall": float(np.mean(mean_rec)),
            "macro_f1": float(np.mean(mean_f1)),
            "per_class_precision": str(mean_prec.tolist()),
            "per_class_recall": str(mean_rec.tolist()),
            "per_class_f1": str(mean_f1.tolist()),
            "n_features_subject": input_size_subject,
            "tr_subject": tr_subject,
        }
    )

print(f"\nAll {len(subjects)} subjects done.")

In [ ]:
if len(all_subjects_results) == 0:
    print("No subject produced valid results. Check missing BOLD files/events and rerun.")
else:
    results_df = pd.DataFrame(all_subjects_results)
    results_df.to_csv(RESULTS_BASE_DIR / "all_subjects_results.csv", index=False)

    print(
        results_df[
            ["subject", "mean_balanced_accuracy", "std_balanced_accuracy", "macro_f1"]
        ].to_string(index=False)
    )

    fig, ax = plt.subplots(figsize=(max(6, len(results_df) * 1.2), 4))
    x = range(len(results_df))
    ax.bar(
        x,
        results_df["mean_balanced_accuracy"] * 100,
        yerr=results_df["std_balanced_accuracy"] * 100,
        capsize=5,
        color="steelblue",
        ecolor="black",
    )
    ax.axhline(25, color="red", linestyle="--", label="Chance (25%)")
    ax.set_xticks(list(x))
    ax.set_xticklabels(results_df["subject"], rotation=45, ha="right")
    ax.set_ylabel("Balanced Accuracy (%)")
    ax.set_title("ANN (subject-specific input -> 64 -> 16 -> 4) | S1 ROI | LORO-CV")
    ax.legend()
    plt.tight_layout()
    plt.savefig(RESULTS_BASE_DIR / "group_balanced_accuracy.png", dpi=150)
    plt.show()

    print(f"\nCSV  -> {(RESULTS_BASE_DIR / 'all_subjects_results.csv').resolve()}")
    print(f"Plot -> {(RESULTS_BASE_DIR / 'group_balanced_accuracy.png').resolve()}")